# 01 · Build the external features and check the joins

Each listing has `(latitude, longitude)`. That is the only key used to attach the external sources.
The feature code lives in `src/external.py`; this notebook runs it on the price sample and checks that every join is sound.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import numpy as np, pandas as pd
from config import CLASSMATE_REPO, SLIM, EXT, OUT
from common import *
from external import build
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
st = pd.read_csv(CLASSMATE_REPO / "data" / "stations_used.csv")   # official IDFM stations (classmate's repo)
print("classmate repo:", CLASSMATE_REPO.exists(), "| slim:", SLIM.exists(), "| external:", EXT.exists())

classmate repo: True | slim: True | external: True


## 1. Price sample (prepared in the classmate's repository)
Entire homes, minimum stay < 30 nights, quoted price 20–1,000 EUR, September 2026 snapshot.

In [2]:
df = pd.read_csv(CLASSMATE_REPO / "data" / "price_features.csv.gz")
print(df.shape); df[["id", "host_id", "latitude", "longitude", "price_eur", "accommodates", "bedrooms", "neighbourhood_cleansed"]].head()

(31044, 26)


,id,host_id,latitude,longitude,price_eur,accommodates,bedrooms,neighbourhood_cleansed
0,3109,3631,48.831910,2.318700,272.50,2,1.0,Observatoire
1,7397,2626,48.859090,2.353150,149.40,4,2.0,Hôtel-de-Ville
2,9952,33534,48.863730,2.370930,177.75,2,1.0,Popincourt
3,11265,41718,48.884940,2.339970,222.86,2,1.0,Buttes-Montmartre
4,11798,44444,48.825241,2.367469,153.50,2,1.0,Gobelins


## 2. Raw external sources: what they contain after cleaning

In [3]:
from external import load_osm, load_dvf, load_iris, load_crime
osm = load_osm(EXT / "osm_paris_poi.json")
print("OSM points by group (an element can count in two groups, e.g. museum = culture and attraction):")
print(osm.group.value_counts().to_string())

OSM points by group (an element can count in two groups, e.g. museum = culture and attraction):
group
food          18618
daily_shop     5481
nightlife      2556
hotel          2084
park           1441
culture        1262
attraction      632


**DVF cleaning rule.** One sale (`id_mutation`) can contain several units and only reports the total price. A price per m² is only meaningful when the sale contains exactly one apartment (plus, possibly, a cellar or parking space). Kept: such sales with surface ≥ 9 m² and 3,000–40,000 EUR/m².

In [4]:
raw = pd.concat([pd.read_csv(EXT / f"dvf_75_{y}.csv.gz", low_memory=False, usecols=["id_mutation", "nature_mutation", "type_local"]) for y in (2022, 2023, 2024, 2025)])
dvf = load_dvf([EXT / f"dvf_75_{y}.csv.gz" for y in (2022, 2023, 2024, 2025)])
print(f"DVF rows: {len(raw):,} | distinct sales: {raw.id_mutation.nunique():,} | single-apartment sales kept: {len(dvf):,}")
print(dvf.eur_m2.describe().round(0).to_string()); print(dvf.date.str[:4].value_counts().sort_index().to_string())

DVF rows: 338,541 | distinct sales: 163,622 | single-apartment sales kept: 112,640
count    112640.0
mean      10600.0
std        3323.0
min        3000.0
25%        8632.0
50%       10167.0
75%       11951.0
max       40000.0
date
2022    33092
2023    26824
2024    24393
2025    28331


In [5]:
polys, meta, inc = load_iris(EXT / "iris_paris.geojson", EXT / "filosofi_iris_2021_disp.zip", EXT / "filosofi_iris_2021_dec.zip")
print("IRIS polygons:", len(polys), "| IRIS types:", meta.iris_type.value_counts().to_dict())
print("Paris IRIS with a published median income:", inc[inc.IRIS.str.startswith("75")].DISP_MED21.notna().sum())
crime = load_crime(EXT / "crime_paris.csv"); print("crime table (2024, rate per 1,000 residents):"); crime.round(1)

IRIS polygons: 992 | IRIS types: {"iris d'habitat": 861, "iris d'activité": 88, 'iris divers': 43}


Paris IRIS with a published median income: 864
crime table (2024, rate per 1,000 residents):


indicateur,crime_burglary_per1000,crime_vandalism_per1000,crime_drug_use_per1000,crime_assault_per1000,crime_theft_per1000,crime_violent_theft_per1000
CODGEO_2026,,,,,,
75101,6.8,23.8,111.9,27.6,325.3,20.9
75102,9.5,16.2,18.1,7.2,123.2,7.5
75103,9.1,10.5,10.7,5.3,63.7,4.7
75104,7.6,13.4,19.6,5.9,97.1,5.0
75105,8.6,7.7,6.4,3.7,50.6,1.6
75106,9.3,7.2,7.3,4.0,69.8,3.1
75107,8.6,7.5,9.7,3.8,48.4,2.6
75108,11.5,17.8,70.8,12.8,150.7,7.3
75109,4.4,13.2,22.6,4.9,92.6,6.2


## 3. Join all sources to the listings

In [6]:
X, G, info, _ = build(df.latitude.values, df.longitude.values, EXT, st)
print(info); print({k: len(v) for k, v in G.items()}, "→", X.shape[1], "external features")

{'osm_points': 32074, 'dvf_sales': 112640, 'iris_polygons': 992, 'points_outside_iris_snapped': 5, 'iris_with_income': 0.9486535240304085, 'points_with_crime': 1.0}
{'stations': 10, 'osm': 29, 'dvf': 7, 'iris_income': 10, 'crime': 6} → 62 external features


## 4. Join checks

1. **Arrondissement agreement.** The arrondissement derived from the IRIS polygon (external) should match the one Airbnb reports.
2. **Coverage.** Share of listings with a value for each source. Missing income is left missing (INSEE does not publish income for some business IRIS); the model handles NaN.

In [7]:
from external import iris_features
ir, _ = iris_features(df.latitude.values, df.longitude.values, polys, meta, inc)
arr_iris = ir.arr_code.str[-2:].astype(int)
arr_airbnb = df.neighbourhood_cleansed.map(pd.Series(range(1, 21), index=["Louvre", "Bourse", "Temple", "Hôtel-de-Ville", "Panthéon", "Luxembourg", "Palais-Bourbon",
    "Élysée", "Opéra", "Entrepôt", "Popincourt", "Reuilly", "Gobelins", "Observatoire", "Vaugirard", "Passy", "Batignolles-Monceau", "Buttes-Montmartre",
    "Buttes-Chaumont", "Ménilmontant"]))
print(f"arrondissement agreement: {(arr_iris == arr_airbnb).mean():.4%}")
cov = pd.Series({k: X[v].notna().all(1).mean() for k, v in G.items()}, name="share of listings with all features of the source")
cov.round(3).to_frame()

arrondissement agreement: 99.9517%


,share of listings with all features of the source
stations,1.000
osm,1.000
dvf,0.993
iris_income,0.949
crime,1.000


In [8]:
X.select_dtypes("number").describe().T[["mean", "50%", "min", "max"]].round(2)

,mean,50%,min,max
ext_st_metro_nearest_km,0.23,0.21,0.00,2.70
ext_st_metro_mean3_km,0.37,0.35,0.15,3.06
ext_st_rer_nearest_km,0.98,0.90,0.01,2.90
ext_st_metro_n_300m,1.08,1.00,0.00,4.00
ext_st_metro_n_500m,2.89,3.00,0.00,8.00
...,...,...,...,...
ext_crime_vandalism_per1000,10.07,9.35,5.39,23.82
ext_crime_drug_use_per1000,19.51,11.38,3.96,111.88
ext_crime_assault_per1000,5.86,4.87,2.42,27.59
ext_crime_theft_per1000,66.08,50.60,21.08,325.33


## 5. Leakage notes
* DVF covers 2022–2025 and crime is 2024: both precede the September 2026 snapshot (and the June 2026 snapshot used for demand targets).
* OSM (Nov 2026) and the station file are later than the snapshots: treated as slow-moving neighbourhood attributes. This is an approximation.
* No external feature uses any Airbnb column other than the coordinates.